In [7]:
import os
import pandas as pd
import numpy as np
import librosa

In [8]:
emotion_map = {
    "ANG": "angry",
    "HAP": "happy",
    "SAD": "sad",
    "NEU": "neutral",
    "FEA": "fear",
    "DIS": "disgust"
}

def classification_emotions(filename):
    for key ,value in emotion_map.items() :
        if key in filename :
            return value


In [9]:
audio_root ="../AudioWAV"

data =[]


In [10]:
def extract_features(file_path, mfcc_n=20, mel_n=128):
    y, sr = librosa.load(file_path, sr=None)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=mfcc_n)
    mfccs_mean = np.mean(mfccs, axis=1)
    mfccs_std = np.std(mfccs, axis=1)

    stft = np.abs(librosa.stft(y))
    chroma = librosa.feature.chroma_stft(S=stft, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=mel_n)
    mel_mean = np.mean(mel, axis=1)

    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_mean = np.mean(zcr, axis=1)

    spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    spec_centroid_mean = np.mean(spec_centroid, axis=1)

    # Pitch
    f0 = librosa.yin(y, fmin=50, fmax=300, sr=sr)
    f0 = f0[~np.isnan(f0)]
    if len(f0) == 0:
        f0_mean, f0_std = 0, 0
    else:
        f0_mean, f0_std = np.mean(f0), np.std(f0)

    # Energy
    rms = librosa.feature.rms(y=y)
    rms_mean = np.mean(rms)
    rms_std = np.std(rms)

    feature_vector = np.concatenate([
        mfccs_mean, mfccs_std,
        chroma_mean,
        mel_mean,
        zcr_mean,
        spec_centroid_mean,
        [f0_mean, f0_std, rms_mean, rms_std]
    ])

    return feature_vector


In [11]:

for root, dirs, files in os.walk(audio_root):
    for file in files:
        if file.endswith(".wav"):
            label = classification_emotions(file)
            if label is not None:
                path = os.path.join(root, file)
                
                data.append({
                    "file_path": path,
                    "label": label,
                    "features":extract_features(path)
                })


/home/malak/.local/lib/python3.10/site-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(


In [12]:
features_array = np.array([d["features"] for d in data])
labels = [d["label"] for d in data]
file_paths = [d["file_path"] for d in data]

df_features = pd.DataFrame(features_array)
df_features["label"] = labels
df_features["file_path"] = file_paths


df_features.to_csv("cremad_features4.csv", index=False)
print("Saved features CSV ✅")

Saved features CSV ✅
